# Lines

**Geometry type:** `line`

Independent line segment pairs — synaptic contact sites, vessel endpoints,
pairwise distance annotations.

1. Generate synthetic line segments (synapse-like pairs)
2. Write to a `.zarrvectors` store
3. Open lazily and inspect metadata
4. Read all segments
5. Spatial bounding-box query
6. Validate


In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import ZarrVectorStore
from zarr_vectors.types.lines import write_lines, read_lines
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "synapses.zarrvectors")
print("Store path:", STORE)


## 1 · Generate synthetic contact-site pairs

In [ ]:
rng = np.random.default_rng(0)
N = 5_000  # synapse pairs

# Synapse midpoints distributed across a 500³ µm volume
midpoints = rng.uniform(50, 450, (N, 3)).astype(np.float32)
# Each synapse is a short segment (1–3 µm) between pre- and post-synaptic sites
offsets = rng.uniform(-1.5, 1.5, (N, 3)).astype(np.float32)

pre_pos  = midpoints - offsets   # (N, 3) pre-synaptic positions
post_pos = midpoints + offsets   # (N, 3) post-synaptic positions

# Flat vertex array: pre_0, post_0, pre_1, post_1, ...
vertices = np.empty((N * 2, 3), dtype=np.float32)
vertices[0::2] = pre_pos
vertices[1::2] = post_pos

# Edge array: each segment connects vertex 2i to vertex 2i+1
edges = np.column_stack([
    np.arange(0, N * 2, 2, dtype=np.int32),
    np.arange(1, N * 2, 2, dtype=np.int32),
])

# Per-vertex attribute: synapse weight
weights = rng.uniform(0, 1, N * 2).astype(np.float32)
synapse_type = np.repeat(rng.integers(0, 4, N).astype(np.int32), 2)

print(f"vertices : {vertices.shape}")
print(f"edges    : {edges.shape}")
print(f"Segment length range: {np.linalg.norm(post_pos - pre_pos, axis=1).min():.2f}"
      f" – {np.linalg.norm(post_pos - pre_pos, axis=1).max():.2f} µm")


## 2 · Write to ZVF store

In [ ]:
write_lines(
    STORE,
    vertices=vertices,
    edges=edges,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    attributes={
        "weight":       weights,
        "synapse_type": synapse_type,
    },
    coordinate_system="RAS",
    axis_units="micrometer",
)
print("Write complete.")


## 3 · Open lazily and inspect metadata

In [ ]:
store = ZarrVectorStore(STORE)

print(f"geometry_type : {store.geometry_type}")
print(f"spatial_dims  : {store.spatial_dims}")
print(f"chunk_shape   : {store.chunk_shape}")
print(f"vertex count  : {store.vertex_count(0):,}  ({N*2} = {N} segments × 2 endpoints)")


## 4 · Read all segments

In [ ]:
result = read_lines(STORE)
print(f"segment_count : {result['segment_count']:,}")
print(f"vertices      : {result['vertices'].shape}")
print(f"edges         : {result['edges'].shape}")
print(f"weight range  : [{result['attributes']['weight'].min():.3f}, "
      f"{result['attributes']['weight'].max():.3f}]")

# Recover (pre, post) pairs
verts = result["vertices"]
edg   = result["edges"]
pre   = verts[edg[:, 0]]
post  = verts[edg[:, 1]]
lengths = np.linalg.norm(post - pre, axis=1)
print(f"\nSegment length: mean={lengths.mean():.2f} µm, "
      f"std={lengths.std():.2f} µm")


## 5 · Return as (N, 2, D) pair array

In [ ]:
result_pairs = read_lines(STORE, return_pairs=True)
pairs = result_pairs["pairs"]          # shape (N, 2, 3)
print(f"pairs shape : {pairs.shape}   (N_segments × 2_endpoints × 3_coords)")
print(f"First pair  : {pairs[0, 0]} → {pairs[0, 1]}")


## 6 · Spatial bounding-box query

In [ ]:
lo = np.array([200.0, 200.0, 200.0])
hi = np.array([300.0, 300.0, 300.0])

bbox_result = read_lines(STORE, bbox=(lo, hi))
print(f"Segments in 100³ µm bbox: {bbox_result['segment_count']:,}")

# A segment is returned if its source vertex (pre-synaptic) falls in the bbox
# Let's verify:
verts_q = bbox_result["vertices"]
edg_q   = bbox_result["edges"]
pre_q   = verts_q[edg_q[:, 0]]
in_bbox = ((pre_q >= lo) & (pre_q <= hi)).all(axis=1)
print(f"Pre-synaptic sites within bbox: {in_bbox.sum()} / {len(in_bbox)}")


## 7 · Validate

In [ ]:
result_v = validate(STORE, level=3)
print(result_v.summary())


## Summary

`line` stores independent segment pairs. Both endpoints of each segment
must lie in the same chunk; use `polyline` for paths that cross chunk
boundaries.

| Step | API |
|------|-----|
| Write | `write_lines(path, vertices, edges, chunk_shape, bin_shape, attributes)` |
| Read all | `read_lines(path)` |
| Read as pairs | `read_lines(path, return_pairs=True)` |
| Bbox query | `read_lines(path, bbox=(lo, hi))` |
